# Lab 01: Foundations & Classical Agents

Kiel University · Agentic AI (infAgAI-01a) · Winter 2026

**Learning objectives.** After this lab you can:

- implement a **simple reflex agent** (condition-action rules) for a toy source-triage task,
- observe precisely **where hand-written rules break**, and name the responsible environment properties in lecture vocabulary,
- replace the rule table with a **single LLM call** (a learned policy) and compare behaviour,
- describe the source-triage agent in **PEAS**, **environment-dimension** and **BDI** vocabulary.

*Estimated effort: about 60 minutes for Parts A to F; Part G is optional. Fill in the gaps (marked with underscores) from Part B onwards; fold-out solutions sit below most cells.*

**Report tasks and hand-in.** Tasks marked 📝 are the report tasks R1 to R3. They come without solutions, and your answers go **directly into this notebook**: each report task is followed by an answer cell marked ✍️ that you fill in (for R2, the completed code cell is part of the answer). There is nothing to upload. The notebook is collected automatically from your folder on the lab server on **Thursday of the week after the lab, at 18:00** (eight days after your lab session); whatever the notebook contains at that moment counts. Save it before then.


## Theory recap: the vocabulary you need today

Everything below is from Session 01.

**Agents, classically.** An **agent** is anything that perceives its environment through *sensors* and acts on it through *actuators* (Russell & Norvig). The **agent function** is the abstract mapping $f: P^* \to A$ from the complete percept sequence to an action; the **agent program** is its concrete implementation on an architecture. A **rational agent** picks the action that maximises the *expected* value of its **performance measure**, given what it has perceived so far; rationality is not omniscience. **PEAS** (Performance measure, Environment, Actuators, Sensors) is the template for specifying a task. **Autonomy** in the classical sense means behaviour driven by the agent's own percepts rather than only by the designer's assumptions; the course also uses the word in a second sense, as the share of control-flow decisions handed to the model (see the autonomy spectrum below).

**Task environments.** Six dimensions: fully vs. partially observable, deterministic vs. stochastic, episodic vs. sequential, static vs. dynamic, discrete vs. continuous, single- vs. multi-agent. The closer a task sits to the "partially observable, stochastic, sequential, dynamic, continuous, multi-agent" end, the harder it is for hand-written rules. Part D puts this to the test.

**Four control architectures, plus learning.** *Simple reflex* (condition-action rules on the current percept only), *model-based reflex* (adds internal state for what it cannot currently see), *goal-based* (looks ahead and plans), *utility-based* (ranks outcomes by a utility function). *Learning* is not a fifth architecture: any of the four can improve its components from experience, and classical agents learned policies long before LLMs existed. Today we build a simple reflex agent. It is fast and trivial to implement, but doomed to fail wherever history matters or the world leaves the designer's assumptions.

**The LLM agent.** The course's working definition: *an LLM agent is a system in which an LLM decides, in a loop, which actions to take next on the way to a goal, calling tools and observing their results.* The LLM is the **policy**, the slot where hand-written decision logic used to live. It is stateless between calls: its working state is the **context**, which the runtime supplies anew on every call.

**The autonomy spectrum.** Single call, augmented call, workflow, agent: each step hands more control-flow decisions from the developer to the model. Autonomy is a dial the designer sets, not a goal, and a chat window is only an interface in front of whichever architecture you chose. This lab sits at the **far-left notch**: bare completions and a rule-based reflex agent. That is the baseline from which every later capability of this course is an explicit, visible addition.

**Classical architectures.** *Deliberative* agents (Shakey / STRIPS, 1971) sense, plan over an explicit symbolic world model, then act: powerful but slow and brittle. *Reactive* agents (Brooks' subsumption architecture, 1986) stack fast stimulus-response layers with no central world model: *"the world is its own best model."* *Hybrid / BDI* agents (Georgeff & Lansky, 1987) keep **beliefs** (current, possibly wrong information), **desires** (goal states, possibly conflicting) and **intentions** (desires the agent has *committed* to; they persist and constrain replanning), with plans drawn from a hand-written library. All three run the **perceive-reason-act loop**, and the loop *closes through the world*: the agent observes what the environment actually did, not what it intended.


❓ **Q:** Distinguish the *agent function* from the *agent program*.
<details><summary>Click for answer</summary>

The agent function is an abstract mathematical mapping from complete percept histories to actions. It specifies the behaviour in every conceivable situation. The agent program is the concrete implementation of (an approximation of) that function, running on a particular architecture. The same distinction applies to LLM agents: the intended behaviour versus the prompt, model and loop code that realise it.

</details>


## Part A: Setup

Same model and client as in Lab 00: `qwen3-30b-a3b-instruct-2507` on the GWDG Academic Cloud, reached through `llm.py` in this folder. This lab also uses `pandas` and `matplotlib`.

The cell below imports everything and checks the connection, like the setup check in Lab 00. If it fails, run the Onboarding notebook first. Everything in this notebook except the LLM cells also works without a connection to the model.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

import llm

MODEL = llm.MODEL   # qwen3-30b-a3b-instruct-2507 on the GWDG Academic Cloud

try:
    resp = llm.chat([{"role": "user", "content": "Reply with exactly one word: ready"}], model=MODEL)
    print(f"Academic Cloud is reachable. Model '{MODEL}' answered:",
          resp["choices"][0]["message"]["content"].strip())
except Exception as e:
    print("Could not talk to the model:", e)
    print("Run the Onboarding notebook first, then re-run this cell.")
    print("Everything except the LLM cells works without a connection.")


<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- The first two `import` lines load the libraries used in this lab: `pandas` (tables, abbreviated `pd`) and `matplotlib.pyplot` (plots, abbreviated `plt`).
- `import llm` loads the file `llm.py` from this folder, the same small client as in Lab 00. Its function `llm.chat()` sends messages to the model on the Academic Cloud. `llm.MODEL` holds the name of the course model; we copy it into our own variable `MODEL` so that every call can say `model=MODEL`.
- The `try` block sends one tiny message to the model. If the call succeeds, the reply is printed. If *anything* goes wrong, for example no API key, no network, or an overloaded server, Python jumps to the `except Exception as e:` block instead of stopping with an error: `e` holds the error message, and we print it together with a hint. You will see this `try`/`except` pattern in every cell that calls the model.
- `resp["choices"][0]["message"]["content"]` is the path to the reply text inside the response (explained in Lab 00): the first of the returned choices, its message, its text. `.strip()` removes blanks and line breaks around it.

</details>


## Part B: Your first bare model call

We start at the **far-left notch of the autonomy dial**: a **single call**, one model invocation, text in, text out, no tools, no loop. You know the ingredients from Lab 00: a `system` message sets standing instructions, the `user` message carries the request, and a low `temperature` makes the output more reproducible.

Warm-up: fill in the two role names, the messages argument and a low temperature such as 0.2.


In [ ]:
SYSTEM_PROMPT = "You are a concise research assistant. Answer in at most two sentences."

messages = [
    {"role": ___, "content": SYSTEM_PROMPT},
    {"role": ___, "content": "In one sentence: what is a simple reflex agent?"},
]

try:
    resp = llm.chat(___, model=MODEL, temperature=___)
    print(resp["choices"][0]["message"]["content"])
except Exception as e:
    print("Model call failed:", e)
    print("If your API key is missing, run the Onboarding notebook first.")


<details>
<summary><b>Click here for the solution</b></summary>

```python
SYSTEM_PROMPT = "You are a concise research assistant. Answer in at most two sentences."

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "In one sentence: what is a simple reflex agent?"},
]

try:
    resp = llm.chat(messages, model=MODEL, temperature=0.2)
    print(resp["choices"][0]["message"]["content"])
except Exception as e:
    print("Model call failed:", e)
    print("If your API key is missing, run the Onboarding notebook first.")
```

</details>


<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `messages` is a Python list with two dictionaries, oldest first. Each dictionary has a `role` and a `content`. The `system` message carries standing instructions that the model treats as coming from the operator ("you are a concise research assistant"); the `user` message is the actual request. Both are plain text; only the role tells the model how to weigh them.
- `llm.chat(messages, model=MODEL, temperature=0.2)` sends the whole list to the model in one request and returns the response as a dictionary. The first argument is the message list; `model=` selects the model; any further keyword argument such as `temperature=` is forwarded to the server unchanged.
- `temperature` controls how much randomness the server uses when it picks each next token. At 0 it always takes the most probable token, so repeated calls give (nearly) the same answer; around 1 it samples freely and answers vary; 0.2 is close to deterministic, right for a short factual answer.
- `resp["choices"][0]["message"]["content"]` is the reply text (see Part A); `print` shows it. The `try`/`except` around the call prints the error instead of stopping the notebook if the server cannot be reached.

</details>


❓ **Q:** How does the single-call end of the autonomy spectrum connect to this lab?
<details><summary>Click for answer</summary>

Lab 01 deliberately starts at the leftmost position: you connect to an open-weight model and issue bare completions: a single call, no tools, no loop. Starting here lets you see every later capability as an explicit, visible addition to this baseline: the agent loop in Unit 2, tools in Unit 3, orchestration in Unit 5. Feeling each notch of the dial prevents treating agent frameworks as black magic. Note that "chatbot" is not a notch on the dial: chat is an interface, and any of the four architectures can sit behind a chat window.

</details>


❓ **Q:** Why does this course run its labs on open-source models rather than commercial products? *(not exam-relevant)*
<details><summary>Click for answer</summary>

Open-weight models on an academic server show more of the machinery than a commercial product does. In a product such as ChatGPT you only see a chat window; the hidden system prompt, tool descriptions and formatting rules the provider adds are invisible. Here, the messages list you build *is* the whole prompt, and you can read the model's weights, tokenizer and chat template if you want to. The GWDG Academic Cloud also costs students nothing, and the same model stays available for reruns. The trade-off is capability: open models fail more often and more visibly than frontier models. For learning purposes this is partly a feature, since the course is centrally concerned with understanding and engineering around failure modes.

</details>


## Part C: A hand-coded simple reflex agent: source triage

Our toy task is **source triage**: deciding which search results are worth reading. The search topic: *"the current state of solid-state battery manufacturing."* A web search on it has returned 20 result records, stored in `data/search_results.csv`:

| column | meaning |
|---|---|
| `domain`, `title`, `snippet` | the **percept**: one search result, as the agent sees it |
| `gold` | the correct **action** (the *reference label*, also called *ground truth*): `KEEP` (useful, trustworthy, relevant) or `DISCARD` |
| `subset` | `easy` (the situations the rule designer anticipated) or `tricky` (see Part D) |

The agent is a **simple reflex agent**: it maps the *current percept only* to an action via ordered condition-action rules. It has no memory, no goals and no model of the world. First, load the data.


In [ ]:
DATA = "data/search_results.csv"

df = pd.read_csv(___)
easy   = df[df["subset"] == ___]
tricky = df[df["subset"] == ___]
print(f"{len(df)} search results: {len(easy)} easy, {len(tricky)} tricky")
df.head()


<details>
<summary><b>Click here for the solution</b></summary>

```python
DATA = "data/search_results.csv"

df = pd.read_csv(DATA)
easy   = df[df["subset"] == "easy"]
tricky = df[df["subset"] == "tricky"]
print(f"{len(df)} search results: {len(easy)} easy, {len(tricky)} tricky")
df.head()
```

</details>


<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `pd.read_csv(DATA)` reads the CSV file and returns a **DataFrame**, pandas' table type: one row per search result, one column per field (`id`, `domain`, `title`, `snippet`, `gold`, `subset`). Every row also has a **row label** (the "index"), which pandas sets to 0, 1, 2, … in file order.
- `df["subset"] == "easy"` compares the `subset` column with the string `"easy"` and gives a column of `True`/`False` values, one per row. Writing that inside `df[...]` keeps exactly the rows with `True`. This is the standard way to filter a table in pandas. The result is a new, smaller DataFrame; `df` itself is unchanged. The filtered rows keep their original labels (the tricky rows are labelled 12 to 19), which does not matter here, because later cells pick rows by position with `iloc`.
- `print(f"...")` is an **f-string**: the expressions inside `{...}` are evaluated and inserted into the text, here the three row counts. `df.head()` returns the first five rows; as the last line of the cell, Jupyter displays it as a table, so you can see what a percept looks like.

</details>


### The rule table

The rules below were written by inspecting the *easy* subset. This is exactly how reflex agents are built in practice: the designer looks at anticipated situations and writes conditions for them. Note the ordering: a trusted origin wins over everything, spam vocabulary discards, quality vocabulary keeps, and the default is conservative. Fill in the gaps.


In [ ]:
TRUSTED_SITES   = ("nature.com", "sciencedirect.com", "ieee.org", "arxiv.org")
SPAM_MARKERS    = ("buy now", "% off", "discount code", "subscribe", "giveaway")
QUALITY_MARKERS = ("peer-reviewed", "journal", "study", "dataset")

def reflex_agent(percept):
    '''Simple reflex agent: condition-action rules on the CURRENT percept only.'''
    domain = percept["domain"].lower()
    text = (percept["title"] + " " + percept["snippet"]).lower()

    if domain.endswith((".edu", ".gov")) or domain in ___:      # rule 1: trusted origin
        return ___
    if any(marker in text for marker in ___):                   # rule 2: commercial spam
        return "DISCARD"
    if any(marker in text for marker in QUALITY_MARKERS):           # rule 3: quality vocabulary
        return ___
    return ___                                                  # default: be conservative

# quick smoke test on the first easy percept
print(reflex_agent(easy.iloc[0].to_dict()), "| expected:", easy.iloc[0]["gold"])


<details>
<summary><b>Click here for the solution</b></summary>

```python
TRUSTED_SITES   = ("nature.com", "sciencedirect.com", "ieee.org", "arxiv.org")
SPAM_MARKERS    = ("buy now", "% off", "discount code", "subscribe", "giveaway")
QUALITY_MARKERS = ("peer-reviewed", "journal", "study", "dataset")

def reflex_agent(percept):
    '''Simple reflex agent: condition-action rules on the CURRENT percept only.'''
    domain = percept["domain"].lower()
    text = (percept["title"] + " " + percept["snippet"]).lower()

    if domain.endswith((".edu", ".gov")) or domain in TRUSTED_SITES:      # rule 1: trusted origin
        return "KEEP"
    if any(marker in text for marker in SPAM_MARKERS):                   # rule 2: commercial spam
        return "DISCARD"
    if any(marker in text for marker in QUALITY_MARKERS):           # rule 3: quality vocabulary
        return "KEEP"
    return "DISCARD"                                                  # default: be conservative

# quick smoke test on the first easy percept
print(reflex_agent(easy.iloc[0].to_dict()), "| expected:", easy.iloc[0]["gold"])
```

</details>


<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `TRUSTED_SITES`, `SPAM_MARKERS` and `QUALITY_MARKERS` are **tuples** of strings (a tuple is a list that cannot be changed afterwards). They are the agent's entire vocabulary: everything it will ever "know" about a search result is whether one of these strings occurs.
- `percept` is one search result as a dictionary with the keys `domain`, `title` and `snippet` (plus `gold` and `subset`, which the agent must not look at). `percept["domain"].lower()` reads the domain and converts it to lowercase, so that `Nature.com` and `nature.com` are treated the same. `text` glues title and snippet into one lowercase string; the rules search in this string.
- `domain.endswith(...)` is true if the string `domain` ends with the given suffix. The double parentheses are not a typo: the outer pair belongs to the function call, the inner pair builds a tuple `(".edu", ".gov")`, which is passed as a single argument. Given a tuple, `endswith` checks every entry and is true if *any* of them matches; `domain.endswith(".edu", ".gov")` without the inner pair would be an error, because the second argument of `endswith` has a different meaning (a start position). `domain in TRUSTED_SITES` is true if the domain is one of the four listed sites. The `or` combines both conditions: **rule 1** fires for any trusted origin and returns `KEEP` immediately. `return` ends the function, so later rules are never checked for this percept.
- `any(marker in text for marker in SPAM_MARKERS)` is a compact way of writing a loop. Read it from the right: `for marker in SPAM_MARKERS` takes the spam markers one after another and calls the current one `marker` (the name is introduced right here, like a loop variable; it exists only inside this expression). For each one, `marker in text` asks whether that string occurs somewhere in `text`, giving `True` or `False`. `any(...)` looks at these results and is `True` if at least one of them is `True`. Written as an ordinary loop, the condition of rule 2 would be:

  ```python
  found = False
  for marker in SPAM_MARKERS:
      if marker in text:
          found = True
  ```

  **Rule 2** then returns `DISCARD`. **Rule 3** does the same with the quality markers and returns `KEEP`. (This `f(x) for x in ...` form inside a function call is called a *generator expression*; you will meet it again in the evaluation function `evaluate` below.)
- If no rule fired, the last line returns `DISCARD`: when in doubt, throw the source away. This default is part of the policy.
- **The order of the rules is part of the design.** A `.gov` page is kept even if its snippet contains "subscribe", because rule 1 is checked before rule 2. Swapping rules 2 and 3 would change the decision for every text that contains both a spam word and a quality word.
- **There is no state anywhere.** The function has no variables that survive from one call to the next, so the same percept always gives the same action and no call can know what an earlier call saw. That is the defining property, and the defining weakness, of a simple reflex agent.
- The smoke test: `easy.iloc[0]` is the first row of the `easy` table (`iloc` selects rows by position, counting from 0), `.to_dict()` turns that row into a dictionary, which is the percept format the agent expects. The printed action should match the gold label of that row.

</details>


### Evaluate on the anticipated situations

We treat this as an *episodic* evaluation for now: each percept is judged independently against `gold`. Complete the **helper function** `evaluate` (a small function that does one recurring job, here: run an agent over a table and count the correct decisions); you will reuse it for every agent in this lab, so that the evaluation code is written only once. It prints one dot per decision (useful once the decisions are LLM calls) and keeps the decisions in `RESULTS` for the comparison table in Part E.


In [ ]:
RESULTS = {}                                  # name -> list of actions, used for the plots in Part E

def evaluate(agent_fn, frame, name):
    actions = []                                 # the agent's decision for each row, in row order
    for _, row in frame.iterrows():
        actions.append(agent_fn(___))        # the percept is one row, as a dict
        print(".", end="", flush=True)           # progress: one dot per decision
    correct = sum(a == g for a, g in zip(actions, frame[___]))
    accuracy = correct / ___
    RESULTS[name] = actions                      # keep the decisions for Part E
    print(f" {name}: {correct}/{len(frame)} correct (accuracy {accuracy:.2f})")
    return accuracy

acc_reflex_easy = evaluate(reflex_agent, easy, "reflex agent, easy subset")


<details>
<summary><b>Click here for the solution</b></summary>

```python
RESULTS = {}                                  # name -> list of actions, used for the plots in Part E

def evaluate(agent_fn, frame, name):
    actions = []                                 # the agent's decision for each row, in row order
    for _, row in frame.iterrows():
        actions.append(agent_fn(row.to_dict()))        # the percept is one row, as a dict
        print(".", end="", flush=True)           # progress: one dot per decision
    correct = sum(a == g for a, g in zip(actions, frame["gold"]))
    accuracy = correct / len(frame)
    RESULTS[name] = actions                      # keep the decisions for Part E
    print(f" {name}: {correct}/{len(frame)} correct (accuracy {accuracy:.2f})")
    return accuracy

acc_reflex_easy = evaluate(reflex_agent, easy, "reflex agent, easy subset")
```

</details>


<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `RESULTS = {}` creates an empty dictionary outside the function. Each evaluation run stores its list of decisions here (the `"KEEP"`/`"DISCARD"` answers, one per row) under a name such as `"LLM agent, easy subset"`, so that Part E can show the decisions later without calling the model again.
- `def evaluate(agent_fn, frame, name):` defines a function with three parameters: `agent_fn` is itself a function (the agent: in Python, functions can be passed around like any other value), `frame` is the DataFrame with the percepts, `name` is a short text such as `"reflex agent, easy subset"`. The function is called four times in this lab (two agents, two subsets), and all four print a line that looks the same; `name` is what tells the lines apart. The same text is also the key under which the decisions are stored in `RESULTS`, so that Part E can look them up again by agent and subset. Because the agent is a parameter, the same function evaluates the rule-based agent now and the LLM agent in Part E.
- `for _, row in frame.iterrows():` goes through the table row by row. `iterrows()` hands out a pair (row label, row) each time; the label is not needed, and `_` is the usual name for a value we ignore.
- `row` is one table row. pandas returns it as its own object that still carries the column names. `row.to_dict()` turns it into an ordinary Python dictionary in which each column name is a key and the row's value in that column is the value, for example `{"domain": "nature.com", "title": "Solid-state batteries: ...", "snippet": "...", "gold": "KEEP", "subset": "easy"}`. That is the percept format `reflex_agent` expects: it reads the fields as `percept["domain"]`, `percept["title"]` and `percept["snippet"]`, so the keys must be exactly the column names.
- `actions = []` creates an empty list before the loop. It collects the agent's decision for every row, in the order of the rows, so that afterwards the first entry belongs to the first row, the second to the second, and so on. Inside the loop, `actions.append(agent_fn(...))` calls the agent with the current percept and adds its answer (`"KEEP"` or `"DISCARD"`) to the end of that list. After the loop, `actions` holds as many decisions as the table has rows. It is used twice: compared with the gold labels to count the correct ones, and stored unchanged in `RESULTS`.
- `print(".", end="", flush=True)` prints a single dot. `end=""` suppresses the line break that `print` normally adds, so the dots form one line; `flush=True` forces the dot onto the screen at once rather than when the cell finishes. With the LLM agent each dot is one answered request, so you can see progress while waiting.
- `zip(actions, frame["gold"])` pairs the first action with the first gold label, the second with the second, and so on. `sum(a == g for a, g in zip(...))` is again the compact loop form from Part C: `for a, g in zip(...)` takes one pair at a time and calls its two parts `a` (the agent's action) and `g` (the gold label); `a == g` is `True` when they agree; `sum(...)` adds all these results up. Since Python counts `True` as 1 and `False` as 0, the sum is the number of correct decisions. As an ordinary loop:

  ```python
  correct = 0
  for a, g in zip(actions, frame["gold"]):
      if a == g:
          correct += 1
  ```
- `accuracy = correct / len(frame)` divides the number of correct decisions by the number of rows (`len(frame)`), which turns the count into a share between 0 and 1. The share, not the count, is what makes the two subsets comparable: 12 correct out of 12 and 6 correct out of 8 are 1.00 and 0.75. `RESULTS[name] = actions` saves the list of decisions under the run's name. The f-string prints a short summary (`{accuracy:.2f}` means: two decimals), and `return accuracy` hands the number back so the calling cell can store it in a variable such as `acc_reflex_easy`.

</details>


## Part D: Where hand-written rules break

A perfect score on the easy subset is no surprise: those are the situations the designer anticipated. The `tricky` subset was hand-picked to be adversarial: every row violates one of the assumptions silently baked into the rule table. Run the agent on it and study each failure. Keep four lecture concepts in hand while you read the output:

- **unanticipated situation**: the world leaves the designer's assumptions (no rule fits, or a rule misfires),
- **history matters**: the correct action depends on earlier percepts, which a simple reflex agent cannot remember,
- **multi-agent, adversarial**: other actors deliberately game the rules (SEO spam),
- **performance measure**: relevance and currency are part of what makes a source worth keeping, but no keyword rule can express them.


In [ ]:
acc_reflex_tricky = evaluate(reflex_agent, tricky, "reflex agent, tricky subset")

print("\nWhere the rules break:")
for _, row in tricky.iterrows():
    action = ___
    if action != ___:
        print(f"- [{row['id']:2d}] [{row['domain']}] {row['title'][:70]}")
        print(f"       agent said {action}, gold label is {row['gold']}")


<details>
<summary><b>Click here for the solution</b></summary>

```python
acc_reflex_tricky = evaluate(reflex_agent, tricky, "reflex agent, tricky subset")

print("\nWhere the rules break:")
for _, row in tricky.iterrows():
    action = reflex_agent(row.to_dict())
    if action != row["gold"]:
        print(f"- [{row['id']:2d}] [{row['domain']}] {row['title'][:70]}")
        print(f"       agent said {action}, gold label is {row['gold']}")
```

</details>


<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- The first line evaluates the reflex agent on the `tricky` rows, exactly as on the easy rows in Part C, and stores the accuracy in `acc_reflex_tricky`. As a side effect, `evaluate` also records the decisions in `RESULTS["reflex agent, tricky subset"]`.
- `print("\nWhere the rules break:")` prints a heading.
- The loop goes through the tricky rows again (see Part C for `iterrows`). For each row it calls the agent and compares the action with the gold label; only mismatches are printed.
- The two `print` lines are f-strings (the `f` before the opening quote). Inside an f-string, **curly braces `{...}`** mark a place where Python evaluates an expression and inserts the result into the text; everything outside the braces is printed as it is. The **square brackets `[...]`** inside are ordinary dictionary access: `row['id']` reads the value stored under the key `id`, exactly like `percept["domain"]` in `reflex_agent`. The key is written with **single quotes** `'id'` only because the whole f-string is already enclosed in double quotes; double quotes inside double quotes would end the string too early. Python treats `'id'` and `"id"` as the same string.
- `{row['id']:2d}`: the part after the colon is a **format specification**. `d` says "print as an integer", `2` says "use at least two characters", so single-digit ids get a leading space and all ids line up in a column.
- `row['title'][:70]` takes the first 70 characters of the title. `[:70]` is a **slice**: everything from the start up to, but not including, position 70. This keeps long titles from wrapping onto a second line.

</details>


> **📝 Report task R1:** For at least four of the failures listed above, name the underlying reason in lecture vocabulary: (a) an *unanticipated situation* outside the designer's assumptions, (b) *history matters*: the simple reflex agent has no memory, (c) an *adversarial multi-agent* environment gaming the rules, or (d) an aspect of the *performance measure* (relevance, currency) that keyword rules cannot express. Justify each in one sentence. Then state which single failure a **model-based reflex agent** (internal state, same rules otherwise) could fix, and why.
> *No solution is provided. Write your answer in the ✍️ cell below.*


✍️ **Your answer to R1.** Replace this line with your answer (Markdown; a bullet point per failure is fine).


❓ **Q:** Why does partial observability force an agent to maintain internal state?
<details><summary>Click for answer</summary>

If a single percept does not reveal the full relevant state of the world, the optimal action can depend on information observed earlier and no longer visible. The agent must therefore accumulate and maintain an internal estimate of the unobserved state. Otherwise it can only implement reflexes on the current percept and will act suboptimally whenever history matters (the duplicate above is exactly such a case). In LLM agents this internal state is the context, which the runtime supplies anew on every call.

</details>


## Part E: Replace the rule table with a learned policy

Now delete the hand-written intelligence and put a **single LLM call** in its place: same interface (`percept -> action`), same task, but the condition-action rules are now *implicit in a learned policy*. Nobody wrote them; nobody can enumerate, verify or individually patch them.

Note carefully what this is and is not: still **one call per percept: no tools, no loop, no memory**. Structurally it remains a simple reflex agent; only the reflex is *learned* instead of *authored*.


In [ ]:
TRIAGE_SYSTEM_PROMPT = (
    "You triage web search results. "
    "The search topic is: 'the current state of solid-state battery manufacturing'. "
    "You see ONE search result. Reply with exactly one word: "
    "KEEP if the source looks trustworthy, relevant to the topic and current, "
    "otherwise DISCARD."
)

def llm_agent(percept):
    '''Same interface as reflex_agent, but the "rule table" is a learned policy.'''
    user_msg = (f"domain: {percept['domain']}\n"
                f"title: {percept['title']}\n"
                f"snippet: {percept['snippet']}")
    resp = llm.chat(
        model=MODEL,
        messages=[{"role": ___, "content": TRIAGE_SYSTEM_PROMPT},
                  {"role": ___, "content": ___}],
        temperature=___,                      # reproducibility over creativity
    )
    answer = resp["choices"][0]["message"]["content"].strip().upper()
    return "KEEP" if ___ else "DISCARD"      # robust parsing of the one-word reply

print(llm_agent(tricky.iloc[0].to_dict()), "| expected:", tricky.iloc[0]["gold"])


<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `TRIAGE_SYSTEM_PROMPT` is the whole "rule table" of this agent, written in English: it states the task, the topic, and the required output (exactly one word). The parentheses let one long string run over several lines; Python joins string pieces that stand next to each other into a single string.
- `def llm_agent(percept):` has the same interface as `reflex_agent`: one percept dictionary in, one action string out. That is what lets `evaluate` run it unchanged.
- `user_msg` is an f-string with three lines (the `\n` inside is a line break): domain, title and snippet, each with a label. This text is all the model learns about the search result.
- `llm.chat(...)` sends two messages, exactly as in Part B: the system prompt as standing instructions and `user_msg` as the request. `temperature=0.0` asks for the most probable answer every time, so that a second run gives the same decisions and your evaluation is reproducible.
- `resp["choices"][0]["message"]["content"]` is the reply text. `.strip()` removes blanks and line breaks around it; `.upper()` converts it to capitals, so that `keep`, `Keep` and `KEEP` all become `KEEP`.
- `"KEEP" if "KEEP" in answer else "DISCARD"` is a **conditional expression**: it evaluates to `"KEEP"` when the word KEEP occurs anywhere in the reply, otherwise to `"DISCARD"`. We test for a substring rather than `answer == "KEEP"` because chat models sometimes add words ("KEEP, the source looks reliable."); an exact comparison would turn every such reply into DISCARD.
- The last line is a smoke test on the first tricky record (`tricky.iloc[0]`, see Part C), printed next to its gold label.

</details>


> **📝 Report task R2:** Complete the gapped cell above (`llm_agent`). In your report, include your completed code and briefly justify two design choices: why temperature $0.0$ for a classifier, and how your parsing handles replies that are not exactly one word.
> *No solution is provided. Leave your completed code in the cell above and write the justification in the ✍️ cell below.*


✍️ **Your answer to R2.** Your completed `llm_agent` cell above is part of the answer; replace this line with your justification of the temperature and the parsing (a few sentences).


### Rules vs. learned policy, side by side

Run both agents on both subsets and plot the comparison. This calls the model once per record (20 calls); with the rate limit of the Academic Cloud expect about two minutes, one dot per decision.


In [ ]:
try:
    acc_llm_easy   = evaluate(___, easy,   "LLM agent, easy subset")
    acc_llm_tricky = evaluate(___, tricky, "LLM agent, tricky subset")

    scores = pd.DataFrame({"reflex agent": [acc_reflex_easy, acc_reflex_tricky],
                           "LLM agent":    [___, ___]},
                          index=["easy subset", "tricky subset"])
    display(scores)                                   # the four accuracies as a table
    scores.plot.bar(ylabel="accuracy", ylim=(0, 1.1), rot=0, color=["#1f4e9c", "#d9820b"],
                    title="Hand-written rules vs. learned policy")
    plt.show()
except Exception as e:
    print("Model call failed:", e)
    print("If your API key is missing, run the Onboarding notebook first.")


<details>
<summary><b>Click here for the solution</b></summary>

```python
try:
    acc_llm_easy   = evaluate(llm_agent, easy,   "LLM agent, easy subset")
    acc_llm_tricky = evaluate(llm_agent, tricky, "LLM agent, tricky subset")

    scores = pd.DataFrame({"reflex agent": [acc_reflex_easy, acc_reflex_tricky],
                           "LLM agent":    [acc_llm_easy, acc_llm_tricky]},
                          index=["easy subset", "tricky subset"])
    display(scores)                                   # the four accuracies as a table
    scores.plot.bar(ylabel="accuracy", ylim=(0, 1.1), rot=0, color=["#1f4e9c", "#d9820b"],
                    title="Hand-written rules vs. learned policy")
    plt.show()
except Exception as e:
    print("Model call failed:", e)
    print("If your API key is missing, run the Onboarding notebook first.")
```

</details>


<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- The two `evaluate` calls run `llm_agent` on both subsets, one model request per row (20 in total), and store the accuracies in `acc_llm_easy` and `acc_llm_tricky`. The dots show progress, one per answered request. The whole cell sits in `try`/`except` because of these calls.
- `pd.DataFrame({...}, index=[...])` builds a **DataFrame**, the pandas table type you met in Part C: a two-dimensional table whose columns may have different data types, with column names and row labels attached. When the data is given as a dictionary, pandas follows a fixed convention: **each key becomes a column name and its list becomes that column**, filled from top to bottom. So `"reflex agent": [acc_reflex_easy, acc_reflex_tricky]` is the first column, with the easy accuracy in the first row and the tricky accuracy in the second. `index=["easy subset", "tricky subset"]` labels the rows, so it must have as many entries as the lists are long. (A list of lists, by contrast, is read row by row: `pd.DataFrame([[1.0, 0.0], [1.0, 0.75]], index=[...], columns=[...])` would put the agents in the rows.)
- `display(scores)` shows the resulting table, agents as columns, subsets as rows, for example `1.00 / 0.00` for the reflex agent and `1.00 / 0.75` for the LLM agent. (`display` is Jupyter's way of rendering an object as it would appear as the last line of a cell; here it is needed because the plot follows.)
- `scores.plot.bar(...)` is pandas' shortcut for a bar chart: one group of bars per row of the table, one bar per column, with a legend taken from the column names. The keyword arguments only label and tidy the plot: `ylabel` names the vertical axis, `ylim=(0, 1.1)` fixes it from 0 to slightly above 1 (accuracy cannot exceed 1), `rot=0` keeps the row names horizontal, `color` gives one colour per agent, `title` sets the title.
- `plt.show()` displays the finished figure below the cell. `plt` is matplotlib's plotting module, imported in the setup cell as `import matplotlib.pyplot as plt`; pandas draws `scores.plot.bar()` with that same library. Without `plt.show()`, Jupyter would usually still show the plot, but together with a technical line such as `<Axes: ...>`.

</details>


Accuracy hides *which* records went wrong. The table below lists every record with the decision of each agent; a ✗ marks a decision that differs from the gold label. No gaps here, just run it and read it next to the failure list from Part D.


In [ ]:
comparison = pd.concat([easy, tricky])[["id", "domain", "gold"]]

for name in ("reflex agent", "LLM agent"):
    if f"{name}, tricky subset" in RESULTS:                       # skip the LLM agent if its run failed
        actions = RESULTS[f"{name}, easy subset"] + RESULTS[f"{name}, tricky subset"]
        comparison[name] = [a if a == g else a + " ✗" for a, g in zip(actions, comparison["gold"])]

comparison


<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `pd.concat([easy, tricky])` stacks the two subsets back into one 20-row table in that order. `[["id", "domain", "gold"]]` keeps only these three columns (a list of column names inside the square brackets selects columns).
- The loop handles the two agents by name. `evaluate` stored each run's decisions in `RESULTS` under names such as `"LLM agent, easy subset"`; the `if` skips an agent whose tricky run is missing, which happens if the model could not be reached.
- `RESULTS[...] + RESULTS[...]` joins the easy decisions and the tricky decisions into one list of 20 (for lists, `+` concatenates), in the same order as the table.
- `comparison[name] = [...]` adds a new column named after the agent. The list comprehension (the compact loop form from Part C) goes through the decisions and gold labels in pairs, calling them `a` and `g`: where they agree, the entry is just the decision; where they differ, a ✗ is appended. `zip` pairs the two lists element by element, as in `evaluate`.
- `comparison` as the last line of the cell makes Jupyter display the table.

</details>


**What you should observe** (results vary between models, quantisations and runs):

- On the **easy** subset the LLM is usually on par with the rules, occasionally slightly worse, because models sometimes over- or under-triage clear cases. Hand-written rules are excellent *inside* their assumptions.
- On the **tricky** subset the LLM typically fixes most of the *semantic* failures: the negation, the adversarial keyword-stuffing, the outdated course notes, and the `.gov` page about wind turbines, which is trustworthy but has nothing to do with the topic. The rule table can only handle situations its designer foresaw when writing the rules; the learned policy is not limited to a list of foreseen cases and can improvise. One record stays debatable: the `ft.com` article (id 18) is relevant and from a serious source, but its snippet only says "subscribe to read", so the model may discard it. Whether a source behind a paywall counts as useful depends on what the agent is for: worthless if it has to read the source, valuable if it only has to list good sources. The gold label took the second view. That is a question about the performance measure, not about the agent.
- The **duplicate (id 17) still gets through**: `llm_agent` sees one percept at a time and remembers nothing between calls, because the LLM is stateless. No amount of prompt cleverness fixes a missing memory. That is the gap that state and the agent loop (Session 02) start to close.


❓ **Q:** Viewed through Russell and Norvig's four control architectures, why is an LLM agent a strange hybrid?
<details><summary>Click for answer</summary>

Behaviourally it resembles a goal-based or utility-based agent: it pursues explicit goals, predicts consequences and trades off options. Structurally, however, none of the required machinery was authored: there are no hand-written condition-action rules, no explicit transition model and no utility function. All of it is implicit in a learned policy. It thus has the competence profile of the top of the hierarchy without the inspectable structure that defined that hierarchy. (Learning itself is not what is new: classical agents learned policies too.)

</details>


## Part F: The triage agent, in classical vocabulary

You have now built two versions of the same agent: one with an *authored* policy (Part C) and one with a *learned* policy (Part E). The lecture (Session 01) reads this task in classical vocabulary, but only for the easy subset. Extend that reading to what you actually built. You should be able to do this in the exam for *any* agent system.


> **📝 Report task R3:** Describe the source-triage agent from Parts C to E in classical vocabulary, in your own words (Markdown, about half a page):
> 1. **PEAS.** Give the specification. Does it differ between `reflex_agent` and `llm_agent`? Justify your answer.
> 2. **Six dimensions.** For the easy subset, the lecture describes the task as fully observable, episodic and static. Go through all six dimensions for the *whole* data set, easy and tricky. For each dimension, say whether that description still holds, and if not, which failure from Part D shows that it does not.
> 3. **BDI reading.** Give beliefs, desires and intentions of `reflex_agent` and of `llm_agent`. What changed structurally when you replaced the rules with a model, and what would have to be added for the agent to have an intention at all?
>
> *No solution is provided. Write your answer in the ✍️ cell below.*


✍️ **Your answer to R3.** Replace this line with your answer (Markdown, about half a page, numbered 1 to 3 as in the task).


❓ **Q:** Define beliefs, desires and intentions, emphasising the functional difference between desires and intentions.
<details><summary>Click for answer</summary>

Beliefs are the agent's current, possibly incorrect, information about the world, updated from percepts. Desires are states the agent would like to bring about; they may be mutually inconsistent and carry no commitment. Intentions are desires the agent has committed to: they persist across deliberation cycles, constrain which further options the agent considers, and trigger plan selection and execution. The commitment is the functional difference: desires motivate deliberation, intentions structure action.

</details>


## Part G: Tuning & exploration (optional)

If time remains. No gaps in this part, just play. Two knobs from Part B steer a bare model call: **temperature** and the **system prompt**. Watch how much behaviour they move without touching a single line of "agent" code.

- **Temperature sweep**: the first cell asks the triage prompt from Part E about the same tricky percept four times per temperature (12 calls, about one minute). Does the KEEP/DISCARD decision flip at higher temperatures? Compare with the creative question in Lab 00: why is a constrained one-word classification far more robust to temperature than open-ended text? Would you still set $T=0$ for the triage step?
- **Personas**: the same percept, three system prompts. The triage decision can flip. The policy lives *in the prompt* as much as in the weights.


In [ ]:
# Temperature sweep on the triage decision: same percept, same prompt, four runs per temperature.
percept = tricky.iloc[0].to_dict()          # the adversarial SEO snippet from Part D (id 13)
user_msg = (f"domain: {percept['domain']}\n"
            f"title: {percept['title']}\n"
            f"snippet: {percept['snippet']}")

for temp in (0.0, 0.7, 1.5):
    votes = []
    for i in range(4):
        try:
            resp = llm.chat([{"role": "system", "content": TRIAGE_SYSTEM_PROMPT},
                             {"role": "user", "content": user_msg}],
                            model=MODEL, temperature=temp)
            votes.append(resp["choices"][0]["message"]["content"].strip().upper()[:10])
        except Exception as e:
            print("Model call failed:", e)
            break
    print(f"T = {temp}: {votes}")


<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `tricky.iloc[0].to_dict()` is the first tricky record (id 13, the keyword-stuffed shop page) as a percept dictionary; `user_msg` lays out its three fields exactly as in `llm_agent`.
- The outer `for temp in (0.0, 0.7, 1.5):` tries three temperatures. For each, `votes = []` starts an empty list, and the inner `for i in range(4):` asks the model four times (`i` only counts the runs; it is not used inside the loop).
- `votes.append(...[:10])` stores the first ten characters of each reply in upper case: enough to read KEEP or DISCARD, and short enough to print the list in one line.
- `break` leaves the inner loop at the first failed call, so a missing key produces one error message instead of twelve. `print(f"T = {temp}: {votes}")` shows the four answers per temperature.

</details>


In [ ]:
percept = tricky.iloc[0].to_dict()          # the adversarial SEO snippet from Part D (id 13)

personas = {
    "sceptical reviewer": "You are a sceptical peer reviewer judging one search result. Reply with one word: KEEP or DISCARD.",
    "eager marketer":     "You love exciting products and hate missing out on anything. Reply with one word: KEEP or DISCARD.",
    "default triage":     TRIAGE_SYSTEM_PROMPT,
}

user_msg = (f"domain: {percept['domain']}\n"
            f"title: {percept['title']}\n"
            f"snippet: {percept['snippet']}")

for name, sys_prompt in personas.items():
    try:
        resp = llm.chat([{"role": "system", "content": sys_prompt},
                         {"role": "user", "content": user_msg}],
                        model=MODEL, temperature=0.0)
        print(f"{name:20s} -> {resp['choices'][0]['message']['content'].strip()[:60]}")
    except Exception as e:
        print("Model call failed:", e)
        break


<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `personas` is a dictionary that maps a short name to a system prompt. The third entry reuses `TRIAGE_SYSTEM_PROMPT` from Part E, so you can compare the two made-up personas against the prompt the agent actually used.
- `user_msg` is the same percept text as in the previous cell (id 13).
- `for name, sys_prompt in personas.items():` goes through the dictionary and gives you key and value in each round. The call sends `sys_prompt` as the system message and `user_msg` as the user message at temperature 0, so any difference between the three answers comes from the system prompt alone.
- In the f-string, `{name:20s}` pads the name to 20 characters so the arrows line up, and `[:60]` cuts the reply after 60 characters.

</details>


❓ **Q:** Name three classical ideas that survive in modern LLM agents, and say what each of them looks like in an LLM agent today.
<details><summary>Click for answer</summary>

The perceive-reason-act loop survives as the agent loop: model call, tool execution, observation appended to the context. Intention persistence survives as explicit plan management in the runtime (the code around the model), which keeps the current plan in the context and discourages constant replanning. Behavioural layering survives as guardrails: cheap, fast, hand-coded checks running beneath expensive model deliberation.

</details>


## Wrap-up

**Takeaways**

- An agent maps percept sequences to actions in an environment. A *simple reflex agent* does so with condition-action rules on the current percept only. Today you watched it fail in the two ways the lecture predicted, cases where history matters (the duplicate) and deliberately misleading sources (the keyword-stuffed spam page), and in two more that the lab added: situations the rule designer did not anticipate, and aspects of the performance measure (relevance, currency) that keyword rules cannot express.
- Replacing the hand-written rule table with a single LLM call swaps an *authored* policy for a *learned* one: it improvises beyond the designer's assumptions (negation, adversarial phrasing), but nobody can enumerate or patch its rules, and it can still be confidently wrong.
- Architecturally, `llm_agent` is *still* a simple reflex agent: one percept in, one action out, no memory. That is why both agents keep the duplicate (id 17): neither can know that it has already seen the same source. The LLM is stateless between calls; fixing this requires state and a loop.
- Autonomy is a **dial**. Today you sat at its far-left notch: bare completions, no tools, no loop. Every later capability of this course is an explicit, visible step to the right.

**Next week:** Session 02 opens the machine and looks at the **agent loop** itself: the LLM call, tool calls, observations, the context as working state, and the stop conditions that keep the loop from running forever.

**Before the hand-in (Thursday of next week, 18:00)**

- [ ] **R1** (Part D): the ✍️ cell holds your classification of at least four reflex-agent failures and names the one failure a model-based reflex agent could fix, and why.
- [ ] **R2** (Part E): the `llm_agent` cell is completed and the ✍️ cell holds your justification of the temperature and parsing choices.
- [ ] **R3** (Part F): the ✍️ cell holds the source-triage agent in classical vocabulary: PEAS, the six environment dimensions on the whole data set, and the BDI reading of both agents.
- [ ] The notebook is saved in your folder on the lab server. It is collected from there automatically; nothing to upload.
